In [1]:
# preparation
import os

In [2]:
# read json into data frame (very slow)
raw_dir = '/nfs/turbo/twitter-decahose/decahose/raw'
df = sqlContext.read.json(os.path.join(raw_dir,'decahose.2022-03-02.p2.bz2'))

22/11/29 21:03:22 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [20]:
# read tweets
tweet = df.select('created_at','extended_tweet.full_text','lang')
# tweet.printSchema()
tweet.show(10, truncate=True)

+--------------------+--------------------+----+
|          created_at|           full_text|lang|
+--------------------+--------------------+----+
|Wed Mar 02 04:51:...|                null|  ja|
|Wed Mar 02 04:51:...|                null|  es|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|house maids servi...|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
+--------------------+--------------------+----+
only showing top 10 rows



In [26]:
# user information
user = df.select('user')
names = df.select('user.id','user.name','user.screen_name', 'user.description')
# names.printSchema()
names.show(10, truncate=True)

+-------------------+--------------------+---------------+-----------------------------------+
|                 id|                name|    screen_name|                        description|
+-------------------+--------------------+---------------+-----------------------------------+
|         1596113701|        (神 ¨̮ 奈)♚✧|  beer_smoke_lv|名古屋のおっぱい代表。 コスプレ ...|
|          581897974|           Majito 🦩|         fgrmr9|               Algún día ⏳\n\n\n...|
|1139392692561932288|      ᴮᴱMy Euphoria⁷|sweetmaknaekook|               The genre is BTS....|
|1341823115836477445|VTuberTweeter | N...|  VTuberTweeter|               ⭐️I auto-retweet ...|
|1480247343676944385|      Jennifer Adams| Ultimatetupman|               Those that have g...|
|1398602686187065344|عايشه عاملات منزل...|1drDqyDU2DHgTf0|               ‏‏‏‏‏عايشه عاملات...|
|1200661454740951040|  camilohernandez012|camiloh54690462|                               null|
|1478253457131847683|     Metro Sports KC| MetroSports_KC|          

In [4]:
# read filtered tweets from sbatch
filter_dir="/nfs/turbo/seas-zhukai/phenology/Twitter/"
# category="antihistamine"
category="pollen"
year="2022"
month="11"
day="01"
df = sqlContext.read.json(os.path.join(filter_dir,category+"/","Spark/",year+"/",month+"/*"))

22/11/30 12:44:42 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [5]:
# read tweets with user info
df_sel = df.select('created_at','user.id','user.screen_name','user.description','extended_tweet.full_text','lang') # screen_name appears to be the handle
df_sel = df_sel.filter(df_sel['full_text'].isNotNull())
df_sel.show(10, truncate=True)

+--------------------+-------------------+---------------+--------------------+--------------------+----+
|          created_at|                 id|    screen_name|         description|           full_text|lang|
+--------------------+-------------------+---------------+--------------------+--------------------+----+
|Tue Nov 01 09:05:...|         3121819423|      STEMbieda|STEM Program Spec...|Treats 🎃 for the...|  en|
|Tue Nov 01 11:27:...|1586122144399990784| sarahkatwest28|Chilled 50 something|@Pollenny1 @OhThe...|  en|
|Tue Nov 01 11:31:...|1563481434165514246|alex19ribeiro84|Audit🔬, Visualiz...|Free NFT Nickelod...|  de|
|Tue Nov 01 11:20:...|1574129364425650176| Nitishkatkade3|Here for the cult...|Impeccable https:...|  en|
|Tue Nov 01 11:20:...|1410184367658483713|     OhTheLies2|Warning! I will c...|@sarahkatwest28 @...|  en|
|Tue Nov 01 17:30:...|1518667113568870401|  PollenToronto|Daily pollen coun...|2022-11-01 - The ...|  en|
|Tue Nov 01 21:34:...|          744452610|Asthma

In [6]:
df_sel.count()

50

# Cleaning

In [7]:
df_lang = df_sel.filter(df_sel['lang']=="en")

In [8]:
# opening the file in read mode
# reading the file
# replacing end of line('/n') with ' ' and
# splitting the text it further when '.' is seen.
keywords = open(os.path.join(filter_dir,category+".txt"), "r").read().split("\n")
keywords[0:5]

['pollen']

In [9]:
#df_word_one=df_sel.filter(df_sel['full_text'].contains('Tavist'))
#df_word_one=df_sel.filter(df_sel['full_text'].rlike(r'\bTavistock\b'))
#df_word_one=df_sel.filter(df_sel['full_text'].rlike(r'\bTavist\b'))
df_word_one=df_lang.filter(df_sel['full_text'].rlike(r'\bBenadryl\b'))
df_word_one.show(5)
#df_word_one.count()

+----------+---+-----------+-----------+---------+----+
|created_at| id|screen_name|description|full_text|lang|
+----------+---+-----------+-----------+---------+----+
+----------+---+-----------+-----------+---------+----+



In [10]:
df_word = None
for keyword in keywords:
    df_word_one = df_lang.filter(df_lang['full_text'].rlike('\\b(?i)'+keyword+'\\b')) # word boundary # case insensitive
    if not df_word:
        df_word = df_word_one
    else:
        df_word = df_word.union(df_word_one)
df_word=df_word.distinct()
df_word.show(10,truncate=True)
df_word.count()

+--------------------+-------------------+---------------+--------------------+--------------------+----+
|          created_at|                 id|    screen_name|         description|           full_text|lang|
+--------------------+-------------------+---------------+--------------------+--------------------+----+
|Tue Nov 01 16:08:...|1484689966559080449|AustinPollenApp|                null|Austin's *Top 10*...|  en|
|Tue Nov 01 21:34:...|          744452610|AsthmaAustralia|Asthma Australia ...|This time of year...|  en|
|Tue Nov 01 06:21:...| 847439135912722432| Disstillmyname|    Work-in-progress|@SolivFlaneur Hey...|  en|
|Tue Nov 01 17:30:...|1518667113568870401|  PollenToronto|Daily pollen coun...|2022-11-01 - The ...|  en|
|Tue Nov 01 16:01:...|1573046392850944005|    BrycewoodWX|Just a lil weathe...|Sunny\nTemperatur...|  en|
|Tue Nov 01 17:22:...|          231302857|AgentPoiznAloha|#Hawaii now in #P...|MY VOICE IS COMIN...|  en|
|Tue Nov 01 09:05:...|         3121819423|    

20

In [11]:
df_word.select("full_text").show(10,truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|full_text                                                                                                                                                                                                                                                                             |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Austin's *Top 10* allergens report for Tuesday 11/1. Cladosporium, elm, ragweed, molds and weeds are moderate. https://t.co/kFDigoWUrn #cladosporium #elm #r

In [12]:
# write to csv
df_word.write.option("header",True).option("delimiter","\t").mode("overwrite").csv(os.path.join(filter_dir,category+"/","CSV/",year+"/",month+"/", day+"/"))
# can then read as pd dataframe in regular Jupyter notebook

In [13]:
df_CC=df_word.filter(df_sel['full_text'].rlike(r'\b(?i)climate\b'))
df_CC.select("full_text").show(10,truncate=False)

+---------+
|full_text|
+---------+
+---------+

